# Sprint 27 — Plots de présentation
## Board NUCLEO-F439ZI : DUAL_MODE (RUL + Faute simultanés)

**Expériences couvertes** : `exp_S27_01` (DUAL_MODE, 200 samples), `exp_S27_02` (latence single vs dual)

**Contenu** :
- Latence single (RUL / MC) vs DUAL — overhead ≈ 0 + budget Gap 2 (100 ms)
- Préservation RUL : RMSE single vs dual vs référence Sprint 26
- Dégradation F1_faute (`FIXME(gap1)` features mixtes)
- Scatter RUL prédit vs vrai (200 samples) + RMSE glissant
- Tableau récapitulatif DUAL_MODE + checklist Gap 2

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path("..")
EXP = ROOT / "experiments"
FIG_DIR = EXP / "figures" / "sprint27"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

def load(exp_id, fname):
    with open(EXP / exp_id / fname) as f:
        return json.load(f)

# exp_S27_01 : DUAL_MODE, 200 samples
s27_01 = load("exp_S27_01", "dual_results.json")
# exp_S27_02 : comparaison latence single vs dual (100 samples / mode)
s27_02_lat = load("exp_S27_02", "latency_comparison.json")

RMSE_REF_S26 = 21.15  # EWC Reg single-mode CMAPSS FD001 (Sprint 26)

print("Donnees chargees OK")
print(f"  S27_01 : RMSE_off={s27_01['metrics_offline']['rmse_rul_offline']:.2f}, "
      f"F1_faute_off={s27_01['metrics_offline']['f1_fault_offline']:.3f}, "
      f"lat={s27_01['metrics_board']['lat_mean_us']:.0f} us, "
      f".bss={s27_01['bss_bytes']:,} B")
print(f"  S27_02 : dual={s27_02_lat['modes']['dual']['lat_mean_us']} us, "
      f"somme_single={s27_02_lat['sum_single_us']} us, overhead={s27_02_lat['overhead_us']} us")

## 1. Latence single vs DUAL — overhead ≈ 0 (Gap 2)

In [ ]:
modes = s27_02_lat["modes"]
labels = ["Single RUL\n(EWC Reg)", "Single MC\n(EWC Multi-class)", "DUAL\n(RUL + MC)"]
lat_mean = [modes["rul_single"]["lat_mean_us"], modes["mc_single"]["lat_mean_us"], modes["dual"]["lat_mean_us"]]
lat_p99 = [modes["rul_single"]["lat_p99_us"], modes["mc_single"]["lat_p99_us"], modes["dual"]["lat_p99_us"]]
colors = ["#2196F3", "#FF5722", "#7B1FA2"]

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(labels))
w = 0.38
b1 = ax.bar(x - w/2, lat_mean, w, color=colors, alpha=0.9, edgecolor="white", label="Latence moyenne")
ax.bar(x + w/2, lat_p99, w, color=colors, alpha=0.45, edgecolor="white", label="Latence P99")
for bar, v in zip(b1, lat_mean):
    ax.text(bar.get_x()+bar.get_width()/2, v+8, f"{v:.0f} us", ha="center", fontsize=10, fontweight="bold")

sum_single = s27_02_lat["sum_single_us"]
overhead = s27_02_lat["overhead_us"]
ax.annotate(
    f"Somme single = {sum_single} us\nDUAL mesure = {modes['dual']['lat_mean_us']:.0f} us\n"
    f"Overhead = {overhead:+.0f} us (sequentiel pur)",
    xy=(2, modes["dual"]["lat_mean_us"]), xytext=(0.7, 540),
    textcoords="data", fontsize=10, color="#4A148C", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="gray"),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#F3E5F5", alpha=0.85))

ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Latence (us)")
ax.set_ylim(0, 720)
ax.set_title("Sprint 27 — Latence single vs DUAL_MODE (NUCLEO-F439ZI)\n"
             f"Budget Gap 2 = 100 ms ; DUAL ~{100_000/modes['dual']['lat_mean_us']:.0f}x sous le budget",
             fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "latency_single_vs_dual.png", bbox_inches="tight")
plt.show()
print(f"Gap 2 satisfait : {s27_02_lat['gap2_satisfied']}")

## 2. Préservation de la régression RUL en mode DUAL

In [ ]:
rmse_single = modes["rul_single"]["rmse_board"]
rmse_dual = modes["dual"]["rmse_rul_offline"]
vals = [RMSE_REF_S26, rmse_single, rmse_dual]
names = ["S26 single\n(reference)", "S27 single RUL\n(100 samples)", "S27 DUAL offline\n(200 samples)"]
colors = ["#43A047", "#2196F3", "#7B1FA2"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, vals, color=colors, width=0.5, edgecolor="white", alpha=0.9)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.2, f"{v:.2f}", ha="center", fontsize=11, fontweight="bold")

ratio = rmse_dual / RMSE_REF_S26
ax.annotate(f"DUAL / S26 = {ratio:.3f}\n(RUL preserve, +{(ratio-1)*100:.0f} %)",
            xy=(2, rmse_dual), xytext=(0.4, rmse_dual*0.6), textcoords="data",
            fontsize=11, color="darkgreen", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="gray"),
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#E8F5E9", alpha=0.85))

ax.set_ylabel("RMSE_RUL (cycles — plus bas = mieux)")
ax.set_ylim(0, max(vals)*1.4)
ax.set_title("Sprint 27 — Preservation RUL en mode DUAL\n"
             "g_ewc_reg lit features[0:5] = top-5 CMAPSS purs -> forward RUL intact",
             fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "rul_preservation_dual.png", bbox_inches="tight")
plt.show()

## 3. Dégradation F1_faute — `FIXME(gap1)` features mixtes

In [ ]:
f1_off = s27_01["metrics_offline"]["f1_fault_offline"]
f1_online = s27_01["metrics_board"]["f1_fault"]
f1_target = 0.50

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(["F1 offline\n(200 samples)", "F1 online\n(dernier sample)"], [f1_off, f1_online],
              color=["#E53935", "#EF9A9A"], width=0.45, edgecolor="white", alpha=0.9)
for bar, v in zip(bars, [f1_off, f1_online]):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")
ax.axhline(f1_target, color="green", linestyle="--", linewidth=1.5, label=f"Seuil attendu = {f1_target:.2f}")

ax.text(0.5, 0.30,
        "FIXME(gap1) — PAS un bug de portage\n"
        "Trame DUAL = 5 features CMAPSS (RUL) + 4 features CWRU (faute) sur 9 slots.\n"
        "g_ewc_mc (entraine sur 9 features CWRU pures) ne recoit que 4/9 slots\n"
        "dans son domaine -> 5 premiers slots hors-distribution.\n"
        "Parite board<->PC exacte. Limitation de CONSTRUCTION du dataset mixte.\n"
        "Resolution : dataset unifie Pronostia (Sprint 28).",
        transform=ax.transAxes, fontsize=9, va="center", ha="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#FFF3E0", edgecolor="#FB8C00", alpha=0.95))

ax.set_ylabel("F1 faute")
ax.set_ylim(0, 0.6)
ax.set_title("Sprint 27 — F1_faute degrade en mode DUAL (FIXME(gap1))", fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
plt.savefig(FIG_DIR / "f1_fault_degradation.png", bbox_inches="tight")
plt.show()

## 4. Régression RUL — prédit vs vrai (200 samples board)

In [ ]:
samples = pd.DataFrame(s27_01["samples"])
rul_true = samples["rul_true"].to_numpy()
rul_pred = samples["rul_pred"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Scatter predit vs vrai ---
ax = axes[0]
ax.scatter(rul_true, rul_pred, s=22, alpha=0.5, color="#7B1FA2", edgecolor="white", linewidth=0.3)
lims = [0, max(rul_true.max(), rul_pred.max())*1.05]
ax.plot(lims, lims, "--", color="gray", linewidth=1.3, label="Ideal (y=x)")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("RUL vrai (cycles)"); ax.set_ylabel("RUL predit (cycles)")
rmse_all = float(np.sqrt(np.mean((rul_pred - rul_true)**2)))
ax.set_title(f"RUL predit vs vrai — DUAL board\nRMSE global = {rmse_all:.2f} cycles", fontweight="bold")
ax.legend(fontsize=9)

# --- RMSE glissant (fenetre 20) ---
ax = axes[1]
win = 20
err2 = (rul_pred - rul_true)**2
roll = np.sqrt(pd.Series(err2).rolling(win, min_periods=1).mean())
ax.plot(samples["i"], roll, color="#7B1FA2", linewidth=1.6)
ax.axhline(rmse_all, color="green", linestyle=":", linewidth=1.4, label=f"RMSE global = {rmse_all:.2f}")
ax.set_xlabel("Index echantillon"); ax.set_ylabel(f"RMSE glissant (fenetre={win})")
ax.set_title("Sprint 27 — Convergence RUL en ligne (DUAL_MODE)", fontweight="bold")
ax.legend(fontsize=9)

fig.suptitle("Sprint 27 — Qualite regression RUL board (200 samples)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "rul_scatter_rolling.png", bbox_inches="tight")
plt.show()

## 5. Tableau récapitulatif DUAL_MODE + checklist Gap 2

In [ ]:
mb = s27_01["metrics_board"]; mo = s27_01["metrics_offline"]
rows = [
    ["RMSE_RUL offline (200 s)", f"{mo['rmse_rul_offline']:.2f} cycles", "< 24.3 (+/-15% vs S26)", "OK"],
    ["F1_faute offline", f"{mo['f1_fault_offline']:.3f}", ">= 0.50", "FIXME(gap1)"],
    ["Latence moyenne DUAL", f"{mb['lat_mean_us']:.0f} us", "< 100 000 us (Gap 2)", "OK"],
    ["Latence P99 DUAL", f"{mb['lat_p99_us']:.0f} us", "< 2 000 us", "OK"],
    ["Overhead vs single", f"{s27_02_lat['overhead_us']:+.0f} us", "~ 0 (sequentiel)", "OK"],
    [".bss firmware", f"{s27_01['bss_bytes']:,} B ({s27_01['bss_bytes']/262144*100:.1f}%)", "< 256 Ko", "OK"],
    ["Forgetting", f"{mb['forgetting']:.3f}", "—", "—"],
]
headers = ["Metrique", "Valeur board", "Critere", "Statut"]

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.axis("off")
status_color = {"OK": "#C8E6C9", "FIXME(gap1)": "#FFCDD2", "—": "#ECEFF1"}
cell_colors = [["#E8EAF6", "white", "white", status_color.get(r[3], "white")] for r in rows]
table = ax.table(cellText=rows, colLabels=headers, cellLoc="center", loc="center", cellColours=cell_colors)
table.auto_set_font_size(False); table.set_fontsize(10); table.scale(1, 2.1)
for j in range(len(headers)):
    table[(0, j)].set_facecolor("#4A148C"); table[(0, j)].set_text_props(color="white", fontweight="bold")
ax.set_title("Sprint 27 — Recapitulatif DUAL_MODE (NUCLEO-F439ZI)", fontweight="bold", fontsize=13, pad=20)
plt.tight_layout()
plt.savefig(FIG_DIR / "dual_mode_summary_table.png", bbox_inches="tight", dpi=150)
plt.show()

## Récapitulatif figures générées

In [ ]:
figs = sorted(FIG_DIR.glob("*.png"))
print(f"\n{len(figs)} figures sauvegardees dans {FIG_DIR}:")
for f in figs:
    print(f"  {f.name}")